In [3]:
import os
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


print("Tensorflow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


Tensorflow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [39]:
df = pd.read_csv('/content/sample_data/ecommerce_complaints_250k_unique.csv')

df.rename(columns={'complaint_text': 'complaint'}, inplace=True)
print('Dataset shape', df.shape)
print('\n columns:')
print(df.columns.tolist())

df.head()


Dataset shape (250000, 3)

 columns:
['complaint_id', 'complaint', 'department']


,complaint_id,complaint,department
0,C038684,An unknown device accessed my account and I ca...,Account & Security
1,C064940,My support request remains unanswered after se...,Customer Support
2,C003955,My registered email was changed despite upload...,Account & Security
3,C120375,My shipment shows a delivery exception while t...,Order & Delivery
4,C172862,The product quality is not advertised after I ...,Product & Seller


In [40]:
#Basic validation

print("missing values\n")
print(df.isnull().sum())
print('\n Duplicate',df.duplicated().sum())
print('\n Deparment Distribution:\n')
print(df['department'].value_counts())

missing values

complaint_id    0
complaint       0
department      0
dtype: int64

 Duplicate 0

 Deparment Distribution:

department
Account & Security       41667
Customer Support         41667
Order & Delivery         41667
Payments & Refunds       41667
Product & Seller         41666
Returns & Replacement    41666
Name: count, dtype: int64


In [41]:
#NLP Preprocessing

X = df['complaint'].astype(str)
y = df['department'].astype(str)

print('Input Samples:')
print(X.head())

print('\n Target Samples:')
print(y.head())

print('\nMaxLen of Complaint')
print(df['complaint'].str.len().max())

Input Samples:
0    An unknown device accessed my account and I ca...
1    My support request remains unanswered after se...
2    My registered email was changed despite upload...
3    My shipment shows a delivery exception while t...
4    The product quality is not advertised after I ...
Name: complaint, dtype: object

 Target Samples:
0    Account & Security
1      Customer Support
2    Account & Security
3      Order & Delivery
4      Product & Seller
Name: department, dtype: object

MaxLen of Complaint
241


In [42]:
#Label Encoding

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print('Department mapping:')

for index, deparment in enumerate(label_encoder.classes_):
  print(f'{index} -> {deparment}')

Department mapping:
0 -> Account & Security
1 -> Customer Support
2 -> Order & Delivery
3 -> Payments & Refunds
4 -> Product & Seller
5 -> Returns & Replacement


In [43]:
#Train test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size = 0.20,
    random_state = 42,
    stratify = y_encoded
)

print("Training samples:",len(X_train))
print("Testing samples:",len(X_test))

Training samples: 200000
Testing samples: 50000


In [46]:
#Tokenizetion

MAX_WORD = 500
MAX_LEN = 300

tokenizer = Tokenizer(
    num_words = MAX_WORD,
    oov_token = '<OOV>'
)

tokenizer.fit_on_texts(X_train)

print("Vocabulary size:", len(tokenizer.word_index))


Vocabulary size: 424


In [47]:
#Convert Text into Sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print('Example text:')
print(X_train.iloc[0])

print('\nSequenc:')
print(X_train_seq[0])

print(tokenizer.word_index)

Example text:
The received item differs from the photos after I checked the stated specifications. and I cannot trust the listing. Please review the seller's representation. because I saved the original listing.

Sequenc:
[2, 88, 26, 203, 82, 2, 411, 10, 3, 93, 2, 69, 107, 6, 3, 16, 218, 2, 29, 4, 32, 2, 71, 83, 7, 3, 226, 2, 47, 29]
{'<OOV>': 1, 'the': 2, 'i': 3, 'please': 4, 'my': 5, 'and': 6, 'because': 7, 'was': 8, 'a': 9, 'after': 10, 'delivery': 11, 'is': 12, 'has': 13, 'need': 14, 'return': 15, 'cannot': 16, 'account': 17, 'so': 18, 'can': 19, 'despite': 20, 'product': 21, 'remains': 22, 'without': 23, 'support': 24, 'not': 25, 'item': 26, 'process': 27, 'while': 28, 'listing': 29, 'which': 30, 'payment': 31, 'review': 32, 'information': 33, 'status': 34, 'provide': 35, 'order': 36, 'details': 37, 'leaving': 38, 'to': 39, 'correct': 40, 'seller': 41, 'replacement': 42, 'package': 43, 'investigate': 44, 'unresolved': 45, 'this': 46, 'original': 47, 'me': 48, 'transaction': 49, 'c

In [61]:
#Padding

# X = df['complaint']
# y = pd.get_dummies(df['department']).values

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

print("Training shape:",X_train_pad.shape)
print("Testing shape:",X_test_pad.shape)

Training shape: (200000, 300)
Testing shape: (50000, 300)


In [62]:
#Model building

model = Sequential()

model.add(
    Embedding(
        input_dim=MAX_WORD,
        output_dim=128,
        input_length= MAX_LEN
    )
)

model.add(
    LSTM(
        32,
        dropout=0.2,
        return_sequences=False
    )
)

model.add(Dense(16, activation='relu'))

model.add(Dense(len(label_encoder.classes_), activation='softmax'))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [63]:
#Compile Model

model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)


In [66]:
#Model Training

y_train_one_hot = tf.one_hot(y_train, depth=len(label_encoder.classes_))

history = model.fit(
    X_train_pad,
    y_train_one_hot,
    validation_data = (X_train_pad, y_train_one_hot),
    epochs =5,
    batch_size = 60,
    verbose = 1
)

Epoch 1/5
3334/3334 ━━━━━━━━━━━━━━━━━━━━ 74s 22ms/step - accuracy: 0.1659 - loss: 1.7919 - val_accuracy: 0.1667 - val_loss: 1.7920
Epoch 2/5
2138/3334 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.1688 - loss: 1.7918

KeyboardInterrupt: 

In [33]:
#Evaluation
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.1667 - loss: 1.7918

Test Loss: 1.7917659282684326
Test Accuracy: 0.16665999591350555


In [36]:
# Debug

for department in label_encoder.classes_:
    print("\n" + "=" * 60)
    print(department)
    print("=" * 60)

    samples = df[df["department"] == department]["complaint"].head(3)

    for text in samples:
        print("-", text)


Account & Security
- The same issue keeps returning after each attempt. Normal login attempts triggered an unexpected security restriction. The latest update shows this problem. Please address the underlying issue and confirm what action will be taken.
- The same issue keeps returning after each attempt. The latest issue is that the recovery process is sending verification requests to unusable contact details. This situation needs attention. A practical resolution is needed instead of another generic update.
- The problem is still active despite the steps already completed. A security review is preventing normal account access. Something in the recent process has gone wrong. The problem should be resolved using the actual transaction or case history.

Customer Support
- The main issue is this. The available information does not provide a reliable resolution. The concern is that an earlier support request has not received a meaningful resolution. Please investigate the actual cause and

In [37]:
print("Vocabulary size:", len(tokenizer.word_index))

print("\nSample vocabulary:")
print(list(tokenizer.word_index.items())[:50])

Vocabulary size: 331

Sample vocabulary:
[('<OOV>', 1), ('the', 2), ('a', 3), ('is', 4), ('issue', 5), ('has', 6), ('not', 7), ('please', 8), ('problem', 9), ('and', 10), ('after', 11), ('what', 12), ('same', 13), ('case', 14), ('update', 15), ('latest', 16), ('information', 17), ('be', 18), ('current', 19), ('to', 20), ('process', 21), ('this', 22), ('resolution', 23), ('status', 24), ('that', 25), ('provide', 26), ('request', 27), ('does', 28), ('records', 29), ('available', 30), ('review', 31), ('have', 32), ('an', 33), ('of', 34), ('unresolved', 35), ('situation', 36), ('earlier', 37), ('concern', 38), ('was', 39), ('actual', 40), ('been', 41), ('happened', 42), ('needs', 43), ('clear', 44), ('needed', 45), ('explain', 46), ('without', 47), ('remains', 48), ('checking', 49), ('reviewed', 50)]
